# Convergence to $\pi$ in Wasserstein and total variation
## NALD with $J=0$, constant $J_a$, and state-dependent $J_s(x)$

We measure how fast the **law of $X_t$** approaches the target, rather than how fast a single
trajectory decorrelates. Concretely: launch $M$ independent chains from a common point mass
$\delta_{x_0}$, and at a sequence of times $t$ estimate the distance between the ensemble
$\mathrm{Law}(X_t)$ and $\pi\propto e^{-U}$.

$$\text{run } M \text{ chains from } x_0,\qquad
  \widehat{\mu}_t=\frac1M\sum_{k=1}^M\delta_{X_t^{(k)}},\qquad
  \text{plot } \ \mathcal{W}_1(\widehat\mu_t,\pi)\ \text{ and }\ \mathrm{TV}(\widehat\mu_t,\pi)\ \text{ vs } t .$$

$\pi$ is represented by $M$ **exact i.i.d. draws** (both targets are exactly sampleable), so no
density estimate or reference chain is involved.

### How the two distances are estimated

Neither $\mathcal{W}_1$ nor TV is directly computable between empirical measures in $\mathbb{R}^3$ at
this sample size, so both are estimated by **slicing** — projecting onto random directions, where each
is exact in one dimension — plus per-coordinate versions for interpretation.

* **Sliced $\mathcal{W}_1$.** For a unit direction $\theta$, the one-dimensional $\mathcal{W}_1$ between
  two equal-size empirical samples is exactly the mean absolute difference of their order statistics.
  We average over $L=64$ fixed directions, drawn once and shared by every method and every time, and
  work in standardised coordinates $y=x/\mathrm{sd}_\pi(x)$ so that all three modes count equally
  rather than the plot being dominated by the widest one. Sliced $\mathcal{W}_1$ is a genuine metric on
  probability measures.
* **Total variation.** TV between two continuous laws cannot be estimated from samples without
  discretising, so we report the **binned** TV on $B=50$ equiprobable reference bins,
  $\mathrm{TV}_B=\tfrac12\sum_b\bigl|\widehat\mu_t(A_b)-\tfrac1B\bigr|$. This is a *lower bound* on the
  true TV for every $B$, increasing to it as $B\to\infty$; it is the standard sample-based proxy. Again
  sliced over the same 64 directions, and also reported per coordinate.

### Two floors, both plotted

Estimated distances cannot go to zero, and there are two distinct reasons — shown explicitly so the
plateaus are not misread as failure to converge.

1. **Finite-sample floor.** Even when $\mathrm{Law}(X_t)=\pi$ exactly, $\widehat\mu_t$ is built from $M$
   samples, so the estimator sits at $O(M^{-1/2})$. We measure it directly, by applying the identical
   estimator to two *independent* exact samples of size $M$. It is drawn as a shaded band.
2. **Discretisation floor.** The chains sample the Euler/splitting invariant law $\pi_h$, not $\pi$.
   Whatever the curves plateau at above the shaded band is that $O(h)$ bias.

Only the descent between these floors is a statement about the dynamics, and that is where the three
perturbations are compared.

In [ ]:
import sys, time, math
sys.path.insert(0, "..")               # nald.py lives at the repository root

import numpy as np
import matplotlib.pyplot as plt
from nald import EllipticLaplace, L1Laplace, hat, nald, J_A, state_scale

%matplotlib inline
np.set_printoptions(precision=4, suppress=True, linewidth=140)
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 120, "font.size": 9,
    "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
    "axes.spines.right": False, "legend.frameon": False, "figure.facecolor": "white",
})
COL = {"J0": "#4C566A", "Ja": "#BF616A", "Js": "#5E81AC"}
LAB = {"J0": r"$J=0$", "Ja": r"$J_a$ (constant)", "Js": r"$J_s(x)$ (state dep.)"}

tgtA = EllipticLaplace(np.diag([9.0, 1.0, 0.25]), 0.1)   # U  = sqrt(x' S^-1 x),  U0 = sqrt(.. + d^2)
tgtB = L1Laplace([2.0, 1.0, 0.4], 0.1)                   # U  = sum |x_i|/b_i,    U0 = sum sqrt(x_i^2+d^2)/b_i
TARGETS = {"A": tgtA, "B": tgtB}
TNAME   = {"A": "elliptical Laplace", "B": r"$\ell^1$ Laplace"}
S_STATE = {k: state_scale(t) for k, t in TARGETS.items()}
SD      = {k: np.sqrt(np.diag(t.cov_exact())) for k, t in TARGETS.items()}

for k in TARGETS:
    print(f"target {k} ({TNAME[k]:<20}) marginal sd = {np.round(SD[k],3)}   s = {S_STATE[k]:.4f}")

In [ ]:
def unit_dirs(L, d, rng):
    v = rng.standard_normal((L, d))
    return v / np.linalg.norm(v, axis=1, keepdims=True)


class Discrepancy:
    r"""Sliced W1 and binned TV between an ensemble and a fixed exact reference sample.

    Sliced quantities use L directions in standardised coordinates y = x / sd_pi, drawn once and
    reused for every method and every time, so the curves are directly comparable.
    TV is the binned TV on B equiprobable reference bins -- a lower bound on the true TV.
    """

    def __init__(self, Xref, sd, L=64, B=50, seed=0):
        self.sd, self.B, self.M = sd, B, Xref.shape[0]
        self.dirs = unit_dirs(L, Xref.shape[1], np.random.default_rng(seed))
        self.Pref = np.sort(Xref/sd @ self.dirs.T, axis=0)      # (M, L) sorted projections
        self.Cref = np.sort(Xref, axis=0)                       # (M, d) sorted coordinates
        q = np.linspace(0, 1, B + 1)[1:-1]
        self.Eproj = np.quantile(self.Pref, q, axis=0)          # (B-1, L) interior bin edges
        self.Ecoor = np.quantile(self.Cref, q, axis=0)          # (B-1, d)

    def _tv(self, sorted_col, edges):
        idx = np.searchsorted(sorted_col, edges)
        cnt = np.diff(np.concatenate(([0], idx, [len(sorted_col)])))
        return 0.5*np.abs(cnt/len(sorted_col) - 1.0/self.B).sum()

    def __call__(self, X):
        P = np.sort(X/self.sd @ self.dirs.T, axis=0)
        Cc = np.sort(X, axis=0)
        return dict(
            sw1 = float(np.abs(P - self.Pref).mean()),                                   # sliced W1
            stv = float(np.mean([self._tv(P[:, l], self.Eproj[:, l]) for l in range(P.shape[1])])),
            w1  = np.abs(Cc - self.Cref).mean(0),                                        # per-coordinate W1
            tv  = np.array([self._tv(Cc[:, j], self.Ecoor[:, j]) for j in range(Cc.shape[1])]),
        )

In [ ]:
def convergence_run(tgt, k, tag, alpha, h, n_steps, M, disc, x0, rec_steps, seed=7, block=25):
    r"""Run M independent chains from the point mass at x0 and evaluate the discrepancies at rec_steps."""
    kw = {"J0": dict(J="none"),
          "Ja": dict(J="const", alpha=alpha),
          "Js": dict(J="state", alpha=alpha, s=S_STATE[k])}[tag]
    X   = np.tile(np.asarray(x0, float), (M, 1))
    out = {kk: [] for kk in ("sw1", "stv", "w1", "tv")}
    done, t0 = 0, time.time()
    for target_step in rec_steps:
        n = target_step - done
        if n > 0:
            _, X = nald(tgt, h=h, n_steps=n, n_chains=M, seed=seed + done, x0=X,
                        store=False, block=block, **kw)
            done = target_step
        d = disc(X)
        for kk in out: out[kk].append(d[kk])
    for kk in out: out[kk] = np.asarray(out[kk])
    out["t"] = np.asarray(rec_steps)*h
    out["wall"] = time.time() - t0
    return out

---
## Setup

Both targets start from the same kind of over-dispersed point mass, $x_0=2\,\mathrm{sd}_\pi$
component-wise — a generic off-axis point that is atypical in every coordinate, so all three modes have
to relax. All three perturbations use the same $M$, the same $x_0$, the same step size and the same
number of $\nabla U_0$ evaluations per unit time, so the horizontal axis is cost as well as time.

$\alpha=4$ for both perturbations: the strongest setting whose discretisation bias was verified
acceptable in the companion notebook.

In [ ]:
M        = 40_000        # independent chains
L_DIR    = 64            # slicing directions
N_BINS   = 50            # equiprobable reference bins for TV
ALPHA    = 4.0
CFG      = {"A": dict(h=0.01,  n_steps=30_000),   # T = 300  (about 5.5 tau_x1 for J = 0)
            "B": dict(h=0.005, n_steps=20_000)}   # T = 100  (about 5.7 tau_x1 for J = 0)

rng  = np.random.default_rng(0)
REF, DISC, FLOOR, X0 = {}, {}, {}, {}
for k, tg in TARGETS.items():
    REF[k]  = tg.sample(M, rng)
    DISC[k] = Discrepancy(REF[k], SD[k], L=L_DIR, B=N_BINS, seed=11)
    X0[k]   = 2*SD[k]
    # finite-sample floor: the same estimator applied to independent exact samples
    fl = [DISC[k](tg.sample(M, rng)) for _ in range(6)]
    FLOOR[k] = {kk: np.mean([f[kk] for f in fl], axis=0) for kk in ("sw1", "stv", "w1", "tv")}
    print(f"target {k}:  x0 = {np.round(X0[k],3)},  T = {CFG[k]['n_steps']*CFG[k]['h']:.0f},  h = {CFG[k]['h']}")
    print(f"            finite-sample floor   sliced W1 = {FLOOR[k]['sw1']:.5f}   sliced TV = {FLOOR[k]['stv']:.5f}")
    print(f"            per-coordinate floor  W1 = {np.round(FLOOR[k]['w1'],4)}   TV = {np.round(FLOOR[k]['tv'],4)}")

In [ ]:
# log-spaced recording times so the early, fast relaxation is resolved too
def rec_grid(n_steps, n=64):
    g = np.unique(np.round(np.logspace(0, np.log10(n_steps), n)).astype(int))
    return np.concatenate(([0], g))

RES = {}
for k, tg in TARGETS.items():
    RES[k] = {}
    rec = rec_grid(CFG[k]["n_steps"])
    print(f"--- target {k}: {M:,} chains x {CFG[k]['n_steps']:,} steps, {len(rec)} recording times ---")
    for tag in ["J0", "Ja", "Js"]:
        r = convergence_run(tg, k, tag, ALPHA, CFG[k]["h"], CFG[k]["n_steps"], M,
                            DISC[k], X0[k], rec)
        RES[k][tag] = r
        print(f"  {tag:<3} final  sliced W1 = {r['sw1'][-1]:.5f}   sliced TV = {r['stv'][-1]:.5f}"
              f"   [{r['wall']:6.1f}s]")

---
## Wasserstein and total variation vs time

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for j, k in enumerate(TARGETS):
    for i, (key, name) in enumerate([("sw1", r"sliced $\mathcal{W}_1$"), ("stv", "sliced TV")]):
        ax = axes[i, j]
        fl = FLOOR[k][key]
        ax.axhspan(0, fl, color="0.88", zorder=0)
        ax.axhline(fl, color="0.55", lw=.8, ls="--", zorder=1)
        for tag in ["J0", "Ja", "Js"]:
            r = RES[k][tag]
            ax.loglog(np.maximum(r["t"], r["t"][1]/2), r[key], color=COL[tag], lw=1.5, label=LAB[tag])
        ax.set_xlabel("simulated time $t$   (= cost: one $\\nabla U_0$ per step, same $h$ for all three)")
        ax.set_ylabel(f"{name}$(\\,\\mathrm{{Law}}(X_t),\\ \\pi\\,)$")
        ax.set_title(f"Target {k} — {TNAME[k]}   ($\\alpha={ALPHA:g}$, $M={M:,}$ chains)")
        ax.text(.015, .06, "finite-sample floor", transform=ax.transAxes, fontsize=7, color="0.4")
        if i == 0 and j == 0: ax.legend(fontsize=8.5, loc="lower left")
fig.suptitle("Convergence of the law of $X_t$ to $\\pi \\propto e^{-U}$ from a common point mass", y=1.005)
fig.tight_layout(); plt.show()

In [ ]:
# How much time each method needs to reach a given accuracy -- the speed-up read off the curves.
def time_to(r, key, level):
    y, t = r[key], r["t"]
    for i in range(1, len(y)):
        if y[i] <= level:
            if y[i-1] <= level: return float(t[i-1])
            if t[i-1] <= 0 or y[i] <= 0 or y[i-1] <= 0: return float(t[i])
            lo, hi = np.log(y[i-1]), np.log(y[i])                 # log-log interpolation
            if hi >= lo: return float(t[i])
            f = (np.log(level) - lo)/(hi - lo)
            return float(np.exp(np.log(t[i-1])*(1-f) + np.log(t[i])*f))
    return np.nan

print("time to reach a multiple of the finite-sample floor (simulated time; speed-up vs J = 0)\n")
TT = {}
for k in TARGETS:
    for key, name in [("sw1", "sliced W1"), ("stv", "sliced TV")]:
        print(f"  target {k}  {name}")
        for mult in [8, 4, 2]:
            lvl = mult*FLOOR[k][key]
            row = {tag: time_to(RES[k][tag], key, lvl) for tag in ["J0", "Ja", "Js"]}
            TT[(k, key, mult)] = row
            base = row["J0"]
            s = "   ".join(f"{tag}: {row[tag]:7.2f}" + (f" ({base/row[tag]:4.2f}x)" if tag != "J0" else "        ")
                           for tag in ["J0", "Ja", "Js"])
            print(f"    {mult}x floor (= {lvl:.4f}):  {s}")
        print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
mults = [8, 4, 2]; w = 0.26
for j, key in enumerate(["sw1", "stv"]):
    ax = axes[j]
    xs = np.arange(len(mults)*len(TARGETS))
    labels = []
    for i, (k, mult) in enumerate([(k, m) for k in TARGETS for m in mults]):
        labels.append(f"{k}\n{mult}x")
        base = TT[(k, key, mult)]["J0"]
        for o, tag in enumerate(["J0", "Ja", "Js"]):
            v = base/TT[(k, key, mult)][tag]
            ax.bar(i + (o-1)*w, v, w, color=COL[tag], label=LAB[tag] if i == 0 else None)
    ax.axhline(1.0, color="0.5", lw=.8, ls=":")
    ax.set_xticks(xs); ax.set_xticklabels(labels, fontsize=7.5)
    ax.set_ylabel("speed-up in time-to-accuracy")
    ax.set_title({"sw1": r"sliced $\mathcal{W}_1$", "stv": "sliced TV"}[key])
    if j == 0: ax.legend(fontsize=8)
fig.suptitle("Time for the law of $X_t$ to reach a fixed multiple of the finite-sample floor", y=1.03)
fig.tight_layout(); plt.show()

---
## Per-coordinate breakdown

The sliced curves mix all directions. Splitting by coordinate shows *which* mode each perturbation
actually accelerates — $x_1$ is the slow one in both targets, $x_3$ the fast one. $\mathcal{W}_1$ here
is in the raw units of each coordinate.

In [ ]:
for k in TARGETS:
    fig, axes = plt.subplots(2, 3, figsize=(12, 6))
    for j in range(3):
        for i, (key, name) in enumerate([("w1", r"$\mathcal{W}_1$"), ("tv", "TV")]):
            ax = axes[i, j]
            fl = FLOOR[k][key][j]
            ax.axhspan(0, fl, color="0.88", zorder=0)
            ax.axhline(fl, color="0.55", lw=.8, ls="--", zorder=1)
            for tag in ["J0", "Ja", "Js"]:
                r = RES[k][tag]
                ax.loglog(np.maximum(r["t"], r["t"][1]/2), r[key][:, j], color=COL[tag], lw=1.4, label=LAB[tag])
            ax.set_title(f"$x_{j+1}$  ({name}),  sd$_\\pi$ = {SD[k][j]:.2f}", fontsize=9)
            ax.set_xlabel("simulated time $t$"); ax.set_ylabel(f"{name} to $\\pi$")
            if i == 0 and j == 0: ax.legend(fontsize=8, loc="lower left")
    fig.suptitle(f"Target {k} — {TNAME[k]}: per-coordinate convergence  ($\\alpha={ALPHA:g}$)", y=1.01)
    fig.tight_layout(); plt.show()

In [ ]:
# the numbers behind the plots
print("sliced W1 and sliced TV at log-spaced times (finite-sample floor quoted per target)\n")
for k in TARGETS:
    fl = FLOOR[k]; ts = RES[k]["J0"]["t"]
    picks = sorted(set(np.round(np.logspace(0, np.log10(len(ts)-1), 8)).astype(int)))
    print(f"  target {k}   floor: sliced W1 {fl['sw1']:.5f},  sliced TV {fl['stv']:.5f}")
    print(f"    {'t':>9}" + "".join(f"{'W1 '+tag:>12}" for tag in ['J0','Ja','Js'])
                          + "".join(f"{'TV '+tag:>12}" for tag in ['J0','Ja','Js']))
    for p in picks:
        print(f"    {ts[p]:>9.2f}" + "".join(f"{RES[k][tag]['sw1'][p]:>12.5f}" for tag in ['J0','Ja','Js'])
                                   + "".join(f"{RES[k][tag]['stv'][p]:>12.5f}" for tag in ['J0','Ja','Js']))
    print()

---
## Reading the plots

* **The descent, not the plateau, is the result.** Every curve flattens out, at the shaded
  finite-sample floor plus whatever $O(h)$ discretisation bias the scheme carries. Where two methods
  plateau at slightly different heights that is a statement about their discretisation error, not
  their mixing; the comparison of interest is the horizontal shift of the descending part, which is
  what the time-to-accuracy table and bar chart quantify.
* **Total variation is a binned lower bound.** With $B=50$ equiprobable bins, $\mathrm{TV}_B$
  under-states the true TV, and a point mass registers as $1-1/B$ rather than $1$ at $t=0$. It is
  monotone in $B$, so the ordering of the three methods is unaffected.
* **The starting point matters and is stated.** All curves begin at the same $\delta_{x_0}$ with
  $x_0=2\,\mathrm{sd}_\pi$; a start closer to the mode compresses the whole picture, and a start along
  a principal axis would specifically penalise $J_s$, whose drift vanishes there.